# Module 3 — Prompt Engineering for Network Ops

**NetOps Co. · CELL-031A · ~35 minutes**

Module 1 produced prose for a human to read. This notebook produces **JSON that code can act on** —
which is the bridge from prompting to agents.

Three techniques, then the thing that makes them reliable.

## Setup — about 60 seconds

Run this once per session. Colab gives you a fresh machine each time, so the clone and
install have to happen again — that is normal, not a mistake.

**Your API key.** Click the 🔑 key icon in the left sidebar, add a secret named
`GEMINI_API_KEY`, and toggle *Notebook access* on. Get a free key at
[aistudio.google.com](https://aistudio.google.com) — no credit card.

Never paste a key into a cell. Notebooks get shared, and the key goes with them.

In [ ]:
!git clone -q https://github.com/telcobytes/netops-genai-course.git 2>/dev/null || (cd netops-genai-course && git pull -q)
!pip install -q google-genai pydantic

import sys, os

# Repo root, resolved rather than hardcoded: the clone Colab just made, the
# checkout this notebook lives in, or the directory it was launched from.
REPO_ROOT = next((p for p in ('/content/netops-genai-course',
                              os.path.abspath(os.path.join(os.getcwd(), '..')),
                              os.getcwd())
                  if os.path.isdir(os.path.join(p, 'data'))), None)
assert REPO_ROOT, 'Could not find the repo root. Run this notebook from inside the checkout.'
sys.path.insert(0, os.path.join(REPO_ROOT, 'data'))

# Key from Colab secrets, with a local fallback so this notebook also runs
# in plain Jupyter.
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    if not os.environ.get('GEMINI_API_KEY'):
        import getpass
        os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')

print('Key loaded:', bool(os.environ.get('GEMINI_API_KEY')))
print('Module 3 — KPI Anomaly Explainer')

### Sanity check — no API key needed

`mock_tools.py` is pure standard library. If this prints numbers, your environment is
working and the rest of the notebook will run.

In [ ]:
import mock_tools
summary = mock_tools.get_cell_kpis('CELL-031A')
print('window     :', summary['window_start'], '->', summary['window_end'])
print('samples    :', summary['sample_count'])
print('rolling avg:', summary['rolling_avg'])
for c in summary['thresholds_crossed']:
    print(f"  CROSSED  {c['metric']} = {c['value']} ({c['comparison']} {c['threshold']})")

---
## 1. What the tool hands you

`get_cell_kpis` does not dump raw counters. It computes rolling averages, deltas and threshold
crossings, and hands the agent a small, already-correct summary.

**Tools compute. Agents reason.** Never make a language model do arithmetic over a long table —
it wastes context and invites confident arithmetic errors.

But note what that does *not* mean. The next cell sends the model `latest`, `rolling_avg` and
`delta`, and holds back one field: `thresholds_crossed`, which the tool already worked out. Send
that and the model is transcribing an answer it was handed — and you cannot tell, because the
answer is right. Held back, it becomes the **answer key** you grade against.

In [ ]:
import json, operator
import mock_tools, nb_viz
from IPython.display import HTML, display

k = mock_tools.get_cell_kpis('CELL-031A')
crossed = {c['metric'] for c in k['thresholds_crossed']}

# 75 / 5 / 95 live in mock_tools.THRESHOLDS and nowhere else. Typing them here
# as well would be a second source of truth: raise the PRB limit in the tool and
# this table would go on highlighting at 75 without erroring.
OPS = {'>': operator.gt, '<': operator.lt}

def past_threshold(row, col):
    if col not in mock_tools.THRESHOLDS:
        return False
    op, limit = mock_tools.THRESHOLDS[col]
    return OPS[op](float(row[col]), limit)

display(HTML(nb_viz.table(
    k['readings'],
    columns=['timestamp','prb_utilization_pct','rrc_setup_success_rate_pct',
             'rrc_drop_rate_pct','active_users'],
    highlight=past_threshold,
    caption='CELL-031A — red cells are past an operational threshold')))

print('rolling avg:', k['rolling_avg'])
print('delta      :', k['delta'])

# What we are willing to send a model: the numbers, NOT the tool's own verdict.
# k['thresholds_crossed'] is held back deliberately -- it is the ANSWER KEY.
# 'latest' has to go in: the rolling average smooths the spike (72.8% PRB
# against a latest reading of 96.3%), so a model given only the average would
# be right to say nothing crossed 75%. Withhold the verdict, send the numbers.
EVIDENCE = json.dumps({'latest': k['latest'],
                       'rolling_avg': k['rolling_avg'],
                       'delta': k['delta']}, indent=2)
print('\nanswer key, withheld from every prompt below:',
      [c['metric'] for c in k['thresholds_crossed']] or 'none')


---
## 2. Zero-shot vs few-shot — the categorization problem

Ask twice, loosely, and watch the category wording drift. That drift is fatal the moment code
has to branch on the answer.

In [ ]:
from llm_client import call_llm

loose = f'Categorize this anomaly in a few words: {EVIDENCE}'
for i in range(3):
    print(f'run {i+1}:', call_llm([{'role':'user','content':loose}]).strip()[:90])

In telecom the useful framing is **domain taxonomy enforcement** — get the model using your
fault categories instead of inventing its own.

But be careful which intervention you credit. Going from the cell above to the cell below
changes *three* things at once: it names the four domains, it pins the reply format, and it
adds two worked examples. Lump those together and you will happily conclude "few-shot fixed
it" when the domain list was doing the work.

So the next cell runs both rungs separately — taxonomy alone, then taxonomy plus examples —
and counts. Whatever the numbers say is your answer, not mine.

In [ ]:
from typing import Literal, get_args
from collections import Counter

# The taxonomy, declared ONCE. The prompts below and the Pydantic model two
# cells down all read from this one declaration, so they cannot drift apart.
# Spelling the four strings out in each place is the two-sources-of-truth
# mistake this module warns about -- easy to make, and silent when it bites.
FaultDomain = Literal['RADIO_ACCESS_INTERFERENCE', 'CAPACITY_PRB_EXHAUSTION',
                      'TRANSPORT_BACKHAUL_JITTER', 'CORE_SIGNALING_REJECT']
DOMAINS = list(get_args(FaultDomain))

# Two worked examples. Note what they are NOT: a near-copy of the cell you are
# classifying. Neither is CELL-031A's answer, and both use only metrics the
# model is actually sent. Examples teach the SHAPE of the reasoning; the domain
# list sets the bounds.
EXAMPLES = '''Example - PRB 38%, setup success 88%, drop 6.1%, users 90 -> CORE_SIGNALING_REJECT
Example - PRB 52%, throughput down 60%, drop 1.2%, users steady -> TRANSPORT_BACKHAUL_JITTER

'''

def rung(examples, runs=3):
    prompt = (f"Categorize into EXACTLY ONE of: {', '.join(DOMAINS)}\n\n"
              + examples
              + f"READINGS: {EVIDENCE}\nReply with the domain name only.")
    answers = [call_llm([{'role':'user','content':prompt}]).strip()
               for _ in range(runs)]
    valid = sum(1 for a in answers if a in DOMAINS)
    for a, n in Counter(answers).most_common():
        flag = '' if a in DOMAINS else '   <- NOT IN THE TAXONOMY'
        print(f'    {a[:46]:<48} x{n}{flag}')
    print(f'    -> {valid}/{runs} usable by code')
    return valid

RUNS = 3          # raise this to 10 when you want evidence rather than a demo

print('RUNG 2  taxonomy named, no examples')
v2 = rung('', RUNS)
print('\nRUNG 3  taxonomy + two worked examples')
v3 = rung(EXAMPLES, RUNS)
print(f'\nThe examples bought you: {v3 - v2:+d} of {RUNS}')

---
## 3. Structured output, enforced — with Pydantic

Asking nicely for JSON gets JSON *most of the time*. Most of the time is fine for prose and
useless for anything your code parses.

Two levels of enforcement, and you want both:

- `json_mode=True` makes the **API** return valid JSON
- a **Pydantic model** makes sure the JSON means what you expect — right fields, right types, right enum

This is the same mechanism that guards tool arguments in Module 10. Learn it here.

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import List
from llm_client import parse_json_response

class AnomalyReport(BaseModel):
    metrics_changed: List[str]
    thresholds_crossed: List[str]
    fault_category: FaultDomain          # the same declaration the prompt used
    summary: str = Field(min_length=20, max_length=300)

# Nothing below retypes a fact that already lives somewhere else: the domains
# come from FaultDomain, the limits come from mock_tools.THRESHOLDS.
# One source of truth per fact.
LIMITS = ', '.join(f'{m} {op} {v:g}'
                   for m, (op, v) in mock_tools.THRESHOLDS.items())

PROMPT = f'''Identify:
1. metrics_changed - which metrics moved most
2. thresholds_crossed - which of these were crossed: {LIMITS}
3. fault_category - EXACTLY ONE of: {', '.join(DOMAINS)}
4. summary - one plain-language sentence, at least 20 characters

Respond ONLY as JSON with those four keys.
READINGS: {{readings}}'''

raw = call_llm([{'role':'user','content': PROMPT.format(readings=EVIDENCE)}],
               json_mode=True)
# json_mode got us something that parses; parse_json_response survives a model
# that wraps it in a code fence. Pydantic decides whether it MEANS what we asked.
report = AnomalyReport(**parse_json_response(raw))
report

Now watch it **reject** something wrong. This is the failure you want — loud, immediate, and free.

In [ ]:
try:
    AnomalyReport(metrics_changed=['prb'], thresholds_crossed=['prb'],
                  fault_category='SOMETHING_I_MADE_UP', summary='short')
except ValidationError as e:
    print(e)

> Two errors caught in microseconds: an invented category, and a summary below the minimum length.
> No API call, no cost, no chance of being wrong about it.

---
## Your turn

### 1. Run the healthy cell — and watch the guard fail

The cell below classifies **CELL-022A**, where nothing is wrong: PRB 57%, setup success
98.7%, drops 1.3%, throughput flat, answer key empty.

You will get a fault anyway. It will pass the `Literal`, it will pass Pydantic, and it will
be false.

**Why:** all four domains are faults. There is no member meaning *"nothing is wrong"*, so on
a healthy cell a schema-conformant answer is wrong **by construction**. The constraint did
not fail to help — it removed the model's ability to say the true thing. Notice that the
loose prompt, the one this module has spent two cells criticising, gets this case right.

`json_mode` buys parseable. Pydantic buys *well-formed*. **Neither buys true.**

### 2. Fix it — give the taxonomy a word for "fine"

- add `'NO_FAULT_DETECTED'` to the `FaultDomain` `Literal` (`DOMAINS` follows on its own)
- tell `PROMPT` when to use it — a member the prompt never mentions is one the model will
  not reach for, so editing the enum alone just moves the failure

Re-run CELL-022A, then re-run CELL-031A: a fix that repairs the healthy cell by making the
model timid on the broken one has moved the failure, not removed it. That regression check
is the half people skip.

Stuck, or want to compare? The repo has the worked answer in
`module03-anomaly-explainer/solution_no_fault.py`, which runs both cells before and after
and scores correctness against the answer key.

**A classifier with no null class will always classify.**

### 3. Raise the sample size

Set `RUNS = 10` in the ladder cell and run it again. Three samples is a demo; ten is closer
to evidence. Write down what each rung bought — Module 10 asks the same question with a
proper harness.

### 4. Add a field the prompt does not know about

Add `confidence: float = Field(ge=0, le=1)` to the model *without* updating the prompt, and
watch Pydantic catch the mismatch.

In [ ]:
# THE HEALTHY CELL. Nothing is wrong here -- watch what comes back anyway.
k2 = mock_tools.get_cell_kpis('CELL-022A')
EVIDENCE2 = json.dumps({'latest': k2['latest'],
                        'rolling_avg': k2['rolling_avg'],
                        'delta': k2['delta']}, indent=2)
print('answer key for CELL-022A:', k2['thresholds_crossed'] or 'EMPTY -- nothing crossed')

raw2 = call_llm([{'role':'user','content': PROMPT.format(readings=EVIDENCE2)}],
                json_mode=True)
report2 = AnomalyReport(**parse_json_response(raw2))

print('\nfault_category     :', report2.fault_category)
print('thresholds_crossed :', report2.thresholds_crossed)
print('summary            :', report2.summary)

# The tell: an empty crossings list and a fault category, in the same validated
# object. Pydantic had no opinion, because "is this true" was never its job.
if not report2.thresholds_crossed and report2.fault_category in DOMAINS:
    print('\n^ Nothing crossed, and it named a fault anyway.')
    print('  Validated. Conformant. False. See Your turn, step 2.')

---
**Next:** the explainer only knows the numbers you hand it. It has no idea NetOps Co. has seen
this pattern before. Module 4 teaches it to remember.